# Metodología

Esta página describe el diseño experimental completo: el pipeline que recibe cada modelo, el
catálogo de modelos con sus espacios de búsqueda, las técnicas de balanceo, la validación cruzada
anidada, los cuatro métodos de optimización y las reglas de reproducibilidad. Todo lo que se
describe está en el paquete `src/` (código con *docstrings* en formato NumPy y pruebas
automáticas en `tests/`), y las tablas de esta página se generan leyendo ese código, de modo que
no pueden desviarse de lo que realmente se ejecutó.

In [1]:
import sys
import warnings
from pathlib import Path

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings('ignore')

import pandas as pd
from IPython.display import Markdown, display

from src import balanceo, config, modelos, optimizacion
from src.analisis import NOMBRES_MODELO
from src.formato import dec, ent, enumerar

pd.set_option('display.max_colwidth', 400)

## 1. El pipeline

Cada combinación se ensambla como un `Pipeline` de `imbalanced-learn` con tres etapas:

1. **Preprocesado** (`src/preprocesamiento.py`): un `ColumnTransformer` que estandariza las
   variables numéricas y booleanas y codifica las categóricas con indicadoras (`OneHotEncoder`
   con `drop='first'` para evitar colinealidad perfecta y `handle_unknown='ignore'` por si un
   nivel aparece solo en validación o prueba). Resultan 78 columnas.
2. **Muestreo** (solo con SMOTE o ADASYN): genera clientes sintéticos de la clase minoritaria.
3. **Modelo**.

Como las tres etapas viven dentro del mismo objeto, en cada pliegue de cada validación cruzada
(externa o interna) el escalado, la codificación y el sobremuestreo se ajustan **solo con las filas
de entrenamiento de ese pliegue**, y las filas de validación nunca se remuestrean. Es la garantía
contra la fuga de información que pide la guía (§3.1) y que faltó en las entregas anteriores.

## 2. Los modelos y sus espacios de búsqueda

Los siete modelos de cada tarea son los de la guía (§7.2), más la EBM como modelo nuevo. Cada uno
se describe una sola vez (`src/modelos.py`) con una rejilla explícita para Grid Search y un espacio
tipado para los otros tres optimizadores (escala logarítmica para C, alpha y tasas de aprendizaje;
enteros en escala logarítmica para vecinos y tamaños de hoja), junto con su complejidad y la
justificación de los rangos.

In [2]:
def describir(espacio):
    partes = []
    for k, (tipo, *args) in espacio.items():
        nombre = k.split('__', 1)[1]
        if tipo == 'cat':
            partes.append(f'{nombre} ∈ {{{", ".join(map(str, args[0]))}}}')
        else:
            escala = {'log': 'log', 'int_log': 'entero, log', 'int': 'entero', 'float': 'uniforme'}[tipo]
            partes.append(f'{nombre} ∈ [{args[0]:g}, {args[1]:g}] ({escala})')
    return '; '.join(partes)

filas = []
for tarea in ('clasificacion', 'regresion'):
    for nombre, spec in modelos.catalogo(tarea, incluir_nuevo=True).items():
        filas.append({'tarea': tarea, 'modelo': NOMBRES_MODELO[nombre],
                      'puntos de la rejilla': spec.n_configuraciones_grid(),
                      'presupuesto por búsqueda': optimizacion.presupuesto_por_defecto(spec.espacio_grid),
                      'espacio de búsqueda': describir(spec.espacio),
                      'ponderación de clases': (f'{spec.peso_clases[0]}={spec.peso_clases[1]}' if spec.peso_clases
                                                else ('—' if tarea == 'regresion' else 'no aplica')),
                      'multi-fidelidad': ('; '.join(f'{fase}: ' + ', '.join(f'{k.split("__", 1)[1]}={v}' for k, v in d.items())
                                                    for fase, d in spec.fidelidad.items()) if spec.fidelidad else '—'),
                      'entrenamiento': spec.complejidad.get('entrenamiento', ''),
                      'inferencia': spec.complejidad.get('inferencia', '')})
catalogo = pd.DataFrame(filas).set_index(['tarea', 'modelo'])
catalogo[['puntos de la rejilla', 'presupuesto por búsqueda', 'espacio de búsqueda', 'ponderación de clases',
          'multi-fidelidad']]

puntos de la rejilla  \
tarea         modelo                                      
clasificacion k-NN                                   32   
              Naive Bayes                            30   
              Regresión logística                    30   
              Árbol de decisión                      48   
              Random Forest                          36   
              XGBoost                                32   
              SVM lineal                             30   
              EBM                                    36   
regresion     k-NN                                   32   
              Ridge                                  30   
              Lasso                                  30   
              Árbol de decisión                      48   
              Random Forest                          36   
              XGBoost                                32   
              SVR lineal                             30   
              EBM                                    36   

                                   presupuesto por búsqueda  \
tarea         modelo                                          
clasificacion k-NN                                       32   
              Naive Bayes                                30   
              Regresión logística                        30   
              Árbol de decisión                          48   
              Random Forest                              36   
              XGBoost                                    32   
              SVM lineal                                 30   
              EBM                                        36   
regresion     k-NN                                       32   
              Ridge                                      30   
              Lasso                                      30   
              Árbol de decisión                          48   
              Random Forest                              36   
              XGBoost                                    32   
              SVR lineal                                 30   
              EBM                                        36   

                                                                                                                                                                                                                                                                            espacio de búsqueda  \
tarea         modelo                                                                                                                                                                                                                                                                              
clasificacion k-NN                                                                                                                                                                                              n_neighbors ∈ [5, 600] (entero, log); weights ∈ {uniform, distance}; p ∈ {1, 2}   
              Naive Bayes                                                                                                                                                                                                                                  var_smoothing ∈ [1e-11, 0.001] (log)   
              Regresión logística                                                                                                                                                                                                                 C ∈ [0.001, 100] (log); l1_ratio ∈ {1.0, 0.0}   
              Árbol de decisión                                                                                                                           max_depth ∈ [3, 30] (entero); min_samples_leaf ∈ [1, 300] (entero, log); ccp_alpha ∈ [1e-06, 0.01] (log); criterion ∈ {gini, entropy}   
              Random Forest                                                                

In [3]:
catalogo[['entrenamiento', 'inferencia']]

entrenamiento  \
tarea         modelo                                                                                                                              
clasificacion k-NN                                                                                 O(1) fuerza bruta; O(n·p·log n) KD/Ball Tree   
              Naive Bayes                                                                                                                O(n·p)   
              Regresión logística                                                                           O(k·n·p) (liblinear, k iteraciones)   
              Árbol de decisión                                                                                                    O(n·p·log n)   
              Random Forest                                                                                                      O(T·n·p·log n)   
              XGBoost                                                                   O(T·d·(n·p + bins·p)) con hist; O(T·d·n·p·log n) exacto   
              SVM lineal                                                                              O(k·n·p) lineal; O(n²·p)–O(n³) con núcleo   
              EBM                  O(B·R·p·(n + bins)) efectos principales + O(p²·bins²) detección de pares (FAST) + O(B·R·K·(n + bins²)) pares   
regresion     k-NN                                                                                 O(1) fuerza bruta; O(n·p·log n) KD/Ball Tree   
              Ridge                                                                                        O(n·p² + p³) cholesky; O(k·n·p) SAGA   
              Lasso                                                                                           O(k·n·p) descenso por coordenadas   
              Árbol de decisión                                                                                                    O(n·p·log n)   
              Random Forest                                                                                                      O(T·n·p·log n)   
              XGBoost                                                                   O(T·d·(n·p + bins·p)) con hist; O(T·d·n·p·log n) exacto   
              SVR lineal                                                                              O(k·n·p) lineal; O(n²·p)–O(n³) con núcleo   
              EBM                  O(B·R·p·(n + bins)) efectos principales + O(p²·bins²) detección de pares (FAST) + O(B·R·K·(n + bins²)) pares   

                                                                                               inferencia  
tarea         modelo                                                                                       
clasificacion k-NN                 O(n·p) por consulta en fuerza bruta; O(p·log n) en árbol con p pequeño  
              Naive Bayes                                                        O(p·clases) por consulta  
              Regresión logística                                                                    O(p)  
              Árbol de decisión                                                         O(d) por consulta  
              Random Forest                                                           O(T·d) por consulta  
              XGBoost                                                                 O(T·d) por consulta  
              SVM lineal                                                   O(p) lineal; O(s·p) con núcleo  
              EBM                                  O(p + K) por consulta: una búsqueda de bin por término  
regresion     k-NN                 O(n·p) por consulta en fuerza bruta; O(p·log n) en árbol con p pequeño  
              Ridge                                                                                  O(p)  
              Lasso                                                                                  O(p)  
              Árbol de decisión                                                

La justificación de cada espacio, tal como está escrita en el código:

In [4]:
for tarea in ('clasificacion', 'regresion'):
    for nombre, spec in modelos.catalogo(tarea, incluir_nuevo=True).items():
        display(Markdown(f'**{NOMBRES_MODELO[nombre]} ({"clasificación" if tarea == "clasificacion" else "regresión"})**: {spec.justificacion}'))

**k-NN (clasificación)**: n_neighbors en escala logarítmica hasta 600: en la Entrega 2 el AUC seguía creciendo en k = 200, así que la rejilla debe cubrir más allá; weights=distance suaviza el voto y p elige entre distancia Manhattan y euclídea. Con p = 78 el árbol KD degenera a fuerza bruta.

**Naive Bayes (clasificación)**: GaussianNB solo tiene var_smoothing, que añade una fracción de la varianza máxima a todas las varianzas para estabilizar; se explora en escala log. No admite class_weight: priors=[0.5, 0.5] es su equivalente exacto (reponderar las clases a prioris iguales).

**Regresión logística (clasificación)**: C es el inverso de la penalización (escala log 1e-3..1e2, como en la Entrega 2, donde el óptimo fue interior). l1_ratio=1 es la penalización L1 (Lasso logístico) y l1_ratio=0 la L2 (Ridge logístico); scikit-learn ≥ 1.8 deprecó el parámetro penalty en favor de l1_ratio. liblinear soporta ambas en binario.

**Árbol de decisión (clasificación)**: max_depth y min_samples_leaf controlan el sobreajuste por dos vías (profundidad y tamaño mínimo de hoja, log hasta 300 ≈ 0,8 % de n); ccp_alpha poda por coste-complejidad en escala log.

**Random Forest (clasificación)**: n_estimators no se optimiza: el error de un bosque decrece de forma monótona con el número de árboles y se estabiliza, así que no tiene un óptimo interior que buscar. Se aplica multi-fidelidad (guía §3.3, «presupuestos parciales: menos árboles»): durante la búsqueda cada configuración se evalúa con 100 árboles y el modelo que se mide en el bucle externo y se guarda se reajusta con 300 (factor de reducción 3). Supuesto: el orden relativo de las configuraciones de max_features, max_depth y min_samples_leaf se conserva con 100 árboles; se contrasta en la sección de cómputo. El presupuesto se gasta en max_features (decorrelación entre árboles), max_depth y min_samples_leaf (regularización de cada árbol).

...

**Árbol de decisión (regresión)**: Igual que el árbol de clasificación; ccp_alpha en una escala menor porque la impureza (MSE) del objetivo, acotado en [0,11; 0,50], es del orden de 1e-3.

**Random Forest (regresión)**: Igual que el bosque de clasificación, incluida la multi-fidelidad (100 → 300 árboles).

**XGBoost (regresión)**: Igual que XGBoost de clasificación, con objetivo reg:squarederror.

**SVR lineal (regresión)**: LinearSVR por la misma razón de coste que LinearSVC. Pérdida ε-insensible cuadrática con el solucionador primal (dual=False): medido el 29-sep sobre un pliegue interno real, la pérdida ε-insensible lineal solo admite el solucionador dual, que no converge en 1.000 iteraciones para ningún C (45-85 s por ajuste con max_iter=10000, sin llegar), mientras que la cuadrática converge en 4-5 iteraciones y < 1 s. epsilon (ancho del tubo insensible) en log entre 1e-4 y 5e-2 porque la desviación típica del objetivo es ≈ 0,067: un tubo mayor que eso ignoraría casi toda la señal.

**EBM (regresión)**: Modelo nuevo, no explicado en clase: Explainable Boosting Machine (Lou, Caruana, Gehrke y Hooker, 2013; paquete interpret de Nori y otros, 2019). Es un modelo aditivo generalizado con interacciones por pares (GA²M): g(E[y]) = β<sub>0</sub> + Σ<sub>j</sub> <i>f</i><sub>j</sub>(<i>x</i><sub>j</sub>) + Σ<sub>ij</sub> <i>f</i><sub>ij</sub>(<i>x</i><sub>i</sub>, <i>x</i><sub>j</sub>). Cada función de forma <i>f</i><sub>j</sub> se aprende por boosting cíclico de árboles de pocas hojas sobre una sola variable a la vez, con una tasa de aprendizaje baja para que el orden de las variables no importe, y se promedia sobre outer_bags submuestras. Es tan legible como una regresión logística (cada variable aporta una curva que se puede graficar) y capta efectos no lineales como el boosting, que es justo lo que faltó en la Entrega 2. learning_rate en log entre 0,008 y 0,1: incluye los valores por defecto de la librería (0,015 en clasificación y 0,04 en regresión), y por debajo de 0,008 cada ajuste tarda el doble (26 s frente a 12 s medidos sobre un pliegue interno de 20.788 filas). max_leaves entre 2 y 4: la EBM aprende en pasos pequeños. min_samples_leaf en log entre 2 y 200 regulariza las funciones de forma. interactions entre 0 (GAM puro, sin pares) y 20: el valor por defecto, 3·p ≈ 234 pares, tardó 112 s por ajuste y en el benchmark sobre entrenamiento las interacciones no aportaron. max_bins en log entre 64 y 1024 fija la resolución de las curvas. Multi-fidelidad como en el Random Forest: la búsqueda usa 2 bolsas externas y el modelo final 8 (las bolsas promedian, no cambian la forma de la función, así que el orden de las configuraciones se conserva). class_weight se implementa con sample_weight (EBMConPesos).

## 3. Balanceo

| Técnica | Implementación | Cuándo no aplica |
|---|---|---|
| Sin balanceo | — | — |
| SMOTE | `imblearn.over_sampling.SMOTE`: interpola nuevos clientes de riesgo alto entre vecinos de la misma clase | Clases ya equilibradas (minoritaria > 40 %), donde no tiene nada que generar |
| ADASYN | Como SMOTE, pero genera más clientes donde la clase minoritaria es más difícil de separar | Igual que SMOTE |
| `class_weight` | Pondera los errores de cada clase en proporción inversa a su frecuencia: `class_weight='balanced'` en logística, árbol, Random Forest y SVM; `priors=[0,5; 0,5]` en Naive Bayes; `scale_pos_weight = negativos/positivos` en XGBoost (calculado con el pliegue); pesos por fila en la EBM | k-NN, cuya predicción es un voto sin función de pérdida que ponderar |

En regresión no hay balanceo de clases (guía §2). Las combinaciones que no aplican se registran en
la tabla maestra con su motivo.

## 4. Validación cruzada anidada

In [5]:
display(Markdown(f"""
- **Bucle externo**: {config.K_EXT} pliegues (estratificados en clasificación), siempre los mismos para
  todos los modelos. Es el único que produce la métrica que se reporta.
- **Bucle interno**: {config.K_INT} pliegues dentro de cada pliegue externo, donde cada optimizador elige los
  hiperparámetros. El mejor puntaje interno se guarda para medir el optimismo de selección, pero
  nunca se reporta como desempeño.
- **Modelo final**: al terminar, la misma búsqueda se repite sobre todo el entrenamiento y el ganador
  se reajusta con todo él. Es el que se guarda, se evalúa una sola vez en la prueba
  ({dec(config.TEST_SIZE * 100, 0)} % de los clientes, reservada desde el principio) y se usa en la interpretabilidad.
- **Métrica principal**: AUC en clasificación, porque no depende del umbral ni de la prevalencia y
  mide lo que necesita una campaña de retención (ordenar a los clientes por riesgo); el umbral y la
  calibración se tratan aparte. RMSE en regresión, porque penaliza más los errores grandes y está en
  las unidades del objetivo.
- **Presupuesto**: cada método evalúa tantas configuraciones como tiene la rejilla de Grid del mismo
  modelo, con un mínimo de {config.PRESUPUESTO_MINIMO}; así los cuatro métodos gastan lo mismo dentro de cada modelo.
"""))


- **Bucle externo**: 5 pliegues (estratificados en clasificación), siempre los mismos para
  todos los modelos. Es el único que produce la métrica que se reporta.
- **Bucle interno**: 3 pliegues dentro de cada pliegue externo, donde cada optimizador elige los
  hiperparámetros. El mejor puntaje interno se guarda para medir el optimismo de selección, pero
  nunca se reporta como desempeño.
- **Modelo final**: al terminar, la misma búsqueda se repite sobre todo el entrenamiento y el ganador
  se reajusta con todo él. Es el que se guarda, se evalúa una sola vez en la prueba
  (20 % de los clientes, reservada desde el principio) y se usa en la interpretabilidad.
- **Métrica principal**: AUC en clasificación, porque no depende del umbral ni de la prevalencia y
  mide lo que necesita una campaña de retención (ordenar a los clientes por riesgo); el umbral y la
  calibración se tratan aparte. RMSE en regresión, porque penaliza más los errores grandes y está en
  las unidades del objetivo.
- **Presupuesto**: cada método evalúa tantas configuraciones como tiene la rejilla de Grid del mismo
  modelo, con un mínimo de 30; así los cuatro métodos gastan lo mismo dentro de cada modelo.


**Por qué 5 pliegues externos y 3 internos.** Con 38.978 clientes de entrenamiento, cada pliegue
externo valida sobre unos 7.800 clientes, lo bastante para que el AUC de un pliegue tenga un error
estándar de unas 5 milésimas, y entrena con el 80 % de los datos, de modo que la estimación apenas
subestima al modelo final entrenado con todos. Diez pliegues duplicarían el coste de las 140 corridas
sin una ganancia apreciable. El bucle interno se repite para cada configuración candidata (30 a 48
por búsqueda) y para cada pliegue externo, así que su número de pliegues multiplica el coste total:
con 3, cada búsqueda entrena con unos 20.800 clientes y valida con unos 10.400, validaciones grandes
que bastan para ordenar las configuraciones, y una corrida completa supone entre 540 y 864 ajustes.
Con 5 pliegues internos el coste crecería un 67 %. La sección de optimizadores mide el precio de esta
elección: el puntaje interno subestima al externo en una milésima de AUC, porque los modelos internos
se entrenan con menos filas, pero ordena las configuraciones igual.

**Entrenamiento, validación y prueba.** La prueba (20 %) se separa al principio y no interviene en
ninguna decisión. Dentro del entrenamiento, la validación es siempre la de la validación cruzada: los
pliegues internos validan para elegir hiperparámetros y los externos, para estimar la
generalización. Las predicciones *fuera de pliegue* (cada cliente de entrenamiento predicho por un
modelo que no lo vio) son el conjunto de validación de la gráfica de entrenamiento, validación y
prueba de la sección de evaluación. Donde un método necesita un conjunto de validación propio (el
*early stopping* de XGBoost en la sección de cómputo), se aparta un 10 % del entrenamiento,
estratificado y con la semilla del proyecto.

## 5. Los cuatro métodos de optimización

| Método | Cómo propone configuraciones | Justificación de sus componentes |
|---|---|---|
| **Grid Search** | Recorre la rejilla explícita completa | Referencia exhaustiva; paraleliza configuraciones × pliegues |
| **Random Search** | Muestrea el espacio tipado con distribuciones continuas (log-uniforme, uniforme, enteros) | Bergstra y Bengio (2012): con pocos hiperparámetros importantes, el muestreo aleatorio cubre mejor cada dimensión que una rejilla del mismo tamaño |
| **Bayesiana (Optuna)** | *Tree-structured Parzen Estimator* (TPE) | **Sustituto**: en lugar de modelar f(x) con un proceso gaussiano, TPE estima dos densidades, ℓ(x) para las configuraciones del cuantil superior (γ = 0,10) y g(x) para el resto, y escala linealmente con el número de ensayos y admite espacios mixtos sin adaptación. **Adquisición**: propone el x que maximiza ℓ(x)/g(x), que equivale a maximizar la mejora esperada (EI). **Arranque**: los primeros 10 ensayos son aleatorios, para que las densidades tengan datos |
| **Genética (DEAP)** | Evoluciona una población de vectores de genes en [0, 1], uno por hiperparámetro | **Selección**: torneo de 3, invariante a la escala del *fitness* (las diferencias de AUC son de milésimas, y la ruleta las aplanaría). **Cruce**: uniforme por gen con probabilidad de intercambio 0,5, aplicado con probabilidad 0,6. **Mutación**: gaussiana con σ = 0,15 en la escala [0, 1], probabilidad 1/n por gen, aplicada con probabilidad 0,3; un hijo idéntico a un genoma ya evaluado se vuelve a mutar. **Elitismo**: el mejor individuo histórico se reinserta en cada generación. **Población**: max(4, min(presupuesto/4, 12)), para tener al menos 3 a 5 generaciones con 30 a 48 evaluaciones. **Diversidad**: se registra por generación (desviación típica media de los genes) para detectar convergencia prematura. Una caché de genomas evita gastar presupuesto en repeticiones |

**Multi-fidelidad (guía §3.3).** Random Forest se busca con 100 árboles y el modelo final se
reajusta con 300; la EBM, con 2 y 8 bolsas externas. Más árboles o más bolsas reducen la varianza del
modelo sin cambiar qué configuración es mejor, así que la búsqueda puede hacerse con la versión
barata. El supuesto se comprueba en la sección de cómputo.

## 6. Reproducibilidad

In [6]:
import importlib.metadata as md
versiones = {p: md.version(p) for p in ('scikit-learn', 'imbalanced-learn', 'xgboost', 'optuna', 'deap',
                                         'interpret-core', 'shap', 'lime', 'statsmodels', 'arch', 'scipy', 'numpy', 'pandas')}
import re
import subprocess
recogidas = subprocess.run([sys.executable, '-m', 'pytest', '--collect-only', '-q'], cwd=RAIZ,
                           capture_output=True, text=True).stdout
n_pruebas = int(re.search(r'(\d+) tests? collected', recogidas).group(1))
display(pd.Series(versiones).to_frame('versión'))
display(Markdown(f"""
- **Semilla única**: `config.SEED = {config.SEED}`, propagada a la partición, a cada estimador
  (`random_state`), a SMOTE y ADASYN, al muestreador TPE de Optuna, al genético y a los bootstrap.
- **Versiones exactas** en `requirements.txt` (arriba, las de las librerías principales).
- **Código congelado**: cada ejecución larga (las 140 corridas, las del modelo nuevo, la comparación con
  la Entrega 2, los experimentos de cómputo) copia `src/` y registra su huella SHA-256, de modo que
  editar el código mientras corre no mezcla versiones. Las corridas de la tabla maestra usaron dos
  versiones documentadas: las 12 primeras se hicieron antes de corregir las especificaciones de SVR y
  SVM, que no convergían, y el resto, con la corrección (la sección de cómputo da los detalles).
- **Tabla maestra** (`resultados/experimentos.parquet`): una fila por corrida con métricas por pliegue,
  hiperparámetros, tiempos, trazas y detalles de cada optimizador; se escribe de forma atómica y con
  un cerrojo, y las corridas se pueden reanudar.
- **{n_pruebas} pruebas automáticas** (`tests/`): ausencia de fugas en la partición y en el pipeline, uniformidad de
  los optimizadores y del registro, y cada contraste estadístico comparado con su definición directa.
"""))

,versión
scikit-learn,1.9.1
imbalanced-learn,0.14.2
xgboost,3.4.1
optuna,5.0.0
deap,1.4.4
interpret-core,0.7.8
shap,0.52.0
lime,0.2.0.1
statsmodels,0.15.0
arch,8.0.0



- **Semilla única**: `config.SEED = 42`, propagada a la partición, a cada estimador
  (`random_state`), a SMOTE y ADASYN, al muestreador TPE de Optuna, al genético y a los bootstrap.
- **Versiones exactas** en `requirements.txt` (arriba, las de las librerías principales).
- **Código congelado**: cada ejecución larga (las 140 corridas, las del modelo nuevo, la comparación con
  la Entrega 2, los experimentos de cómputo) copia `src/` y registra su huella SHA-256, de modo que
  editar el código mientras corre no mezcla versiones. Las corridas de la tabla maestra usaron dos
  versiones documentadas: las 12 primeras se hicieron antes de corregir las especificaciones de SVR y
  SVM, que no convergían, y el resto, con la corrección (la sección de cómputo da los detalles).
- **Tabla maestra** (`resultados/experimentos.parquet`): una fila por corrida con métricas por pliegue,
  hiperparámetros, tiempos, trazas y detalles de cada optimizador; se escribe de forma atómica y con
  un cerrojo, y las corridas se pueden reanudar.
- **72 pruebas automáticas** (`tests/`): ausencia de fugas en la partición y en el pipeline, uniformidad de
  los optimizadores y del registro, y cada contraste estadístico comparado con su definición directa.
